In [40]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

In [52]:
input_folder = "input images"
output_folder = "output images"

os.makedirs(output_folder, exist_ok=True)

In [53]:
def classify_biscuit(contour):
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    
    if perimeter == 0:
        return "Unknown"
    
    circularity = 4 * np.pi * area / (perimeter * perimeter)

    # Threshold (you can tune this)
    if circularity > 0.75:
        return "Intact Biscuit"
    else:
        return "Broken Biscuit"

In [61]:
import cv2
import numpy as np
import os

# ✅ FIXED folder names (match your setup)
input_folder = "input images"
output_folder = "output images"

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):

    if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    image_path = os.path.join(input_folder, filename)
    image = cv2.imread(image_path)

    if image is None:
        print("Could not read:", filename)
        continue

    # Resize (optional but good for consistency)
    image = cv2.resize(image, (600, 800))
    output = image.copy()

    # ---------------- PREPROCESSING ----------------
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    lower_biscuit = np.array([5, 30, 45])
    upper_biscuit = np.array([38, 255, 255])

    mask = cv2.inRange(hsv, lower_biscuit, upper_biscuit)

    # Improve mask (noise removal)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    # ---------------- CONTOUR DETECTION ----------------
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    total = 0

    for contour in contours:

        area = cv2.contourArea(contour)

        if area < 700:
            continue

        total += 1

        x, y, w, h = cv2.boundingRect(contour)

        perimeter = cv2.arcLength(contour, True)
        if perimeter == 0:
            continue

        # ---------------- FEATURE EXTRACTION ----------------
        circularity = 4 * np.pi * area / (perimeter * perimeter)

        aspect_ratio = w / float(h)

        hull = cv2.convexHull(contour)
        hull_area = cv2.contourArea(hull)
        if hull_area == 0:
            continue

        solidity = area / hull_area

        rect_area = w * h
        extent = area / rect_area

        missing_area = hull_area - area
        missing_ratio = missing_area / hull_area

        approx = cv2.approxPolyDP(contour, 0.03 * perimeter, True)
        corners = len(approx)

        # ---------------- AUTO SHAPE DETECTION ----------------
        if circularity > 0.7:
            shape = "round"
        else:
            shape = "square"

        # ---------------- CLASSIFICATION ----------------
        if shape == "round":

            if circularity > 0.75 and solidity > 0.90:
                label = "Intact Biscuit"
                color = (0, 255, 0)
            else:
                label = "Broken Biscuit"
                color = (0, 0, 255)

        elif shape == "square":

            if (
                area > 11000 and
                extent > 0.72 and
                solidity > 0.95 and
                0.75 <= aspect_ratio <= 1.35 and
                missing_ratio < 0.06 and
                corners >= 4
            ):
                label = "Intact Biscuit"
                color = (0, 255, 0)
            else:
                label = "Broken Biscuit"
                color = (0, 0, 255)

        else:
            label = "Biscuit"
            color = (255, 0, 0)

        # ---------------- DRAW RESULTS ----------------
        cv2.rectangle(output, (x, y), (x + w, y + h), color, 2)
        cv2.drawContours(output, [contour], -1, color, 2)

        cv2.putText(
            output,
            label,
            (x, y - 8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            2
        )

        # Debug info (helps tuning)
        print(
            filename,
            "| Area:", int(area),
            "| Extent:", round(extent, 2),
            "| Solidity:", round(solidity, 2),
            "| Aspect:", round(aspect_ratio, 2),
            "| Missing:", round(missing_ratio, 2),
            "| Corners:", corners,
            "| Shape:", shape,
            "=>", label
        )

    # ---------------- SAVE OUTPUT ----------------
    output_path = os.path.join(output_folder, "result_" + filename)
    cv2.imwrite(output_path, output)

    print(f"{filename}: Total biscuits detected = {total}")
    print("Saved:", output_path)
    print("-" * 50)

print("Finished processing all images.")

IMG_1.jpeg | Area: 22097 | Extent: 0.95 | Solidity: 0.98 | Aspect: 1.06 | Missing: 0.02 | Corners: 4 | Shape: round => Intact Biscuit
IMG_1.jpeg | Area: 21538 | Extent: 0.95 | Solidity: 0.98 | Aspect: 1.05 | Missing: 0.02 | Corners: 4 | Shape: round => Intact Biscuit
IMG_1.jpeg | Area: 4614 | Extent: 0.54 | Solidity: 0.94 | Aspect: 1.15 | Missing: 0.06 | Corners: 4 | Shape: square => Broken Biscuit
IMG_1.jpeg | Area: 13350 | Extent: 0.83 | Solidity: 0.94 | Aspect: 1.6 | Missing: 0.06 | Corners: 4 | Shape: square => Broken Biscuit
IMG_1.jpeg | Area: 20552 | Extent: 0.93 | Solidity: 0.98 | Aspect: 1.06 | Missing: 0.02 | Corners: 4 | Shape: round => Intact Biscuit
IMG_1.jpeg | Area: 8414 | Extent: 0.63 | Solidity: 0.9 | Aspect: 0.66 | Missing: 0.1 | Corners: 5 | Shape: square => Broken Biscuit
IMG_1.jpeg | Area: 4173 | Extent: 0.66 | Solidity: 0.93 | Aspect: 1.01 | Missing: 0.07 | Corners: 5 | Shape: square => Broken Biscuit
IMG_1.jpeg | Area: 10437 | Extent: 0.63 | Solidity: 0.94 | Aspec